# Data Coverage

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter
from coverage_functions import coverage_calculator, plot_time_series, plot_time_spacing

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

### Calculating Weekly Data Coverage

In [ ]:
# Calculate active weeks
active_weeks_dict = {}
for loc_id, df in sales_and_menu_data.items():
    active_weeks_dict[loc_id] = (df
                                 .resample('W')
                                 .size()
                                 .to_frame(name='W')
                                 .query('0 < W')
                                 .index
                                 .tz_localize(None)
                                 .to_period('W')
                                 .tolist())

### Weekly Data Coverage Visual

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Sort active_weeks_dict by the earliest start date of each restaurant
sorted_active_weeks = dict(
    sorted(active_weeks_dict.items(), key=lambda item: min(week.start_time for week in item[1]))
)

# Create the figure and axes with a larger size
coverage_fig, ax = plt.subplots(figsize=(12, 8))

# Lists to store values for efficient plotting
y_values = []
x_starts = []
x_ends = []

# Loop through sorted active weeks
for loc_id, active_weeks in sorted_active_weeks.items():
    # Get the promo date
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date'].tz_convert('UTC')

    # Collect start and end times for efficient plotting
    for week in active_weeks:
        y_values.append(loc_id)
        x_starts.append(week.start_time)
        x_ends.append(week.end_time)

    # Plot the promotional item as a red circle with a refined marker
    ax.plot(promo_datetime, loc_id, 'o', color='red', markersize=6, alpha=0.6, label='Promotion Date' if loc_id == list(sorted_active_weeks.keys())[0] else "")
  

# Plot all horizontal lines at once with an improved line style
ax.hlines(y=y_values, xmin=x_starts, xmax=x_ends, colors='steelblue', lw=3, alpha=0.8, label='Active Weeks')

# Set labels and title with enhanced fonts
ax.set_title('Weekly Sales Presence', fontsize=26, weight='bold', pad=32)
plt.suptitle("(Across Thirty Restaurants)", fontsize=20, y=0.875, x=.5)
ax.set_xlabel('Date', fontsize=21, labelpad=8)

# Format the x-axis to show major ticks for years and minor ticks for half-years
ax.xaxis.set_major_locator(mdates.YearLocator())  # Major ticks every year
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=6))  # Minor ticks every 6 months
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Year labels
ax.tick_params(axis='x', which='major', length=4, labelsize=14)
ax.tick_params(axis='x', which='minor', length=1)  # Shorter ticks for half-years

# Add weekly grid lines
ax.xaxis.set_minor_locator(mdates.MonthLocator())  # Add back weekly grid lines
ax.grid(visible=True, which='major', axis='x', linestyle='--', alpha=0.45)
ax.grid(visible=True, which='minor', axis='x', linestyle=':', alpha=0.25)  # Dotted grid lines for weekly

# Remove y-axis ticks and labels
ax.yaxis.set_ticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)  # Remove the left spine for a cleaner look

# Set bounds
ax.set_xlim(pd.Timestamp('2013-01-01'), pd.Timestamp('2023-07-01'))

# Add a legend with a refined position
ax.legend(loc='upper left', fontsize=16)

# Adjust layout for tightness and save the figure
coverage_fig.tight_layout()
plt.savefig('Data_Coverage_Enhanced_Alt5.png', dpi=400, bbox_inches='tight')
plt.show()


In [ ]:
# Visualizing with gaps for inactive weeks
coverage_fig, ax = plt.subplots(figsize=(14, 8))

# Loop through every active week within a single restaurant
for loc_id, active_weeks in active_weeks_dict.items():

    # Index into the promotional items for this restaurant
    promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_convert('UTC')

    # For every active week
    for week in active_weeks:

        # Place a blue dot
        ax.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    # Place a red circle for the promotional item
    ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
ax.set_xlabel('Date')
ax.set_ylabel('Restaurant ID')

# Figure
coverage_fig.tight_layout()

# Save
plt.savefig('Data Coverage.png', bbox_inches='tight')
plt.savefig("datacoverage.png")
plt.show()

In [ ]:
# # Sample data (not directly used in hlines but can be used if needed)
# x = np.linspace(0, 10, 100)
# y = np.exp(x)

# fig, ax = plt.subplots(figsize=(14, 8))

# # Define the positions of the rectangles (different horizontal locations)
# rectangle_positions = [2, 4, 6]  # x-center positions for rectangles

# # Define rectangle properties (same size and color)
# rect_width = 0.5
# rect_height = 0.5
# rect_color = 'black'

# # Draw the base line
# ax.hlines(y=1, xmin=0, xmax=10, colors='black', lw=2)

# # Draw the rectangles, zoomed lines, and connecting lines
# for i, rect_x_center in enumerate(rectangle_positions):
#     # Calculate rectangle bottom position
#     rect_y_bottom = i + 1.75  # Adjust vertical position for each level

#     # Draw rectangle
#     rect = patches.Rectangle(
#         (rect_x_center - rect_width / 2, rect_y_bottom),
#         rect_width,
#         rect_height,
#         linewidth=1,
#         edgecolor=rect_color,
#         facecolor='none'
#     )
#     ax.add_patch(rect)

#     # Define zoomed x-range based on rectangle's position
#     zoom_xmin = rect_x_center - rect_width / 2
#     zoom_xmax = rect_x_center + rect_width / 2

#     # Draw zoomed line at next level
#     zoom_level_y = i + 2  # Levels start from y=2
#     ax.hlines(y=zoom_level_y, xmin=zoom_xmin, xmax=zoom_xmax, colors='blue', lw=2)

#     # Draw diagonal lines from rectangle corners to the edges of the zoomed line
#     rect_top_y = rect_y_bottom + rect_height
#     ax.plot([rect_x_center - rect_width / 2, zoom_xmin], [rect_top_y, zoom_level_y], color='gray', linestyle='--')
#     ax.plot([rect_x_center + rect_width / 2, zoom_xmax], [rect_top_y, zoom_level_y], color='gray', linestyle='--')

#     # For the first rectangle, draw lines connecting from the base line to the rectangle
#     if i == 0:
#         ax.plot([rect_x_center - rect_width / 2, rect_x_center - rect_width / 2], [1, rect_top_y], color='gray', linestyle='--')
#         ax.plot([rect_x_center + rect_width / 2, rect_x_center + rect_width / 2], [1, rect_top_y], color='gray', linestyle='--')

# # Optionally, add one more zoom level within the last rectangle
# # Further zoom into the last rectangle
# last_rect_x_center = rectangle_positions[-1]
# small_rect_width = rect_width / 2  # Smaller rectangle width
# small_rect_height = rect_height
# small_rect_y_bottom = len(rectangle_positions) + 1.75  # Position higher

# # Draw the small rectangle inside the last rectangle's area
# small_rect = patches.Rectangle(
#     (last_rect_x_center - small_rect_width / 2, small_rect_y_bottom),
#     small_rect_width,
#     small_rect_height,
#     linewidth=1,
#     edgecolor=rect_color,
#     facecolor='none'
# )
# ax.add_patch(small_rect)

# # Draw the final zoomed line
# final_zoom_level_y = len(rectangle_positions) + 2  # Next level
# final_zoom_xmin = last_rect_x_center - small_rect_width / 2
# final_zoom_xmax = last_rect_x_center + small_rect_width / 2
# ax.hlines(y=final_zoom_level_y, xmin=final_zoom_xmin, xmax=final_zoom_xmax, colors='blue', lw=2)

# # Draw diagonal lines from small rectangle to the final zoomed line
# small_rect_top_y = small_rect_y_bottom + small_rect_height
# ax.plot([final_zoom_xmin, final_zoom_xmin], [small_rect_top_y, final_zoom_level_y], color='gray', linestyle='--')
# ax.plot([final_zoom_xmax, final_zoom_xmax], [small_rect_top_y, final_zoom_level_y], color='gray', linestyle='--')

# # Set plot limits and labels
# ax.set_xlim(0, 10)
# ax.set_ylim(0, final_zoom_level_y + 1)
# ax.set_xlabel('X-axis')
# ax.set_ylabel('Levels')

# # Customize y-ticks to show level descriptions
# ax.set_yticks([1] + [i + 2 for i in range(len(rectangle_positions) + 1)])
# ax.set_yticklabels(['Base Level'] + [f'Zoom {i+1}' for i in range(1, len(rectangle_positions) + 2)])

# plt.title('Zoomed Views with Highlighted Sections and Connections')
# plt.show()

### Data Density: Entire Set, Before Promo, and After Promo

In [ ]:
%store -r restaurant_data_unprocessed
# Determine timezones
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
    
# Timezone mapping
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_coverage_list_4m = []
data_coverage_list_b2m = []
data_coverage_list_a2m = []
data_coverage_list_all = []
data_coverage_list_before = []
data_coverage_list_after = []

for loc_id, df in sales_and_menu_data.items():
    row_4m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['4mo'])
    row_b2m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['b2m'])
    row_a2m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['a2m'])
    row_all = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['all'])
    row_before = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['bef'])
    row_after = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['aft'])
    
    data_coverage_list_4m.append(row_4m)
    data_coverage_list_b2m.append(row_b2m)
    data_coverage_list_a2m.append(row_a2m)
    data_coverage_list_all.append(row_all)
    data_coverage_list_before.append(row_before)
    data_coverage_list_after.append(row_after)

# Create data frame
data_coverage_4m = pd.DataFrame(data_coverage_list_4m).sort_values('4mo_12H_cover', ascending=False).set_index('loc_id')
restaurants_by_4m_coverage = data_coverage_4m.index.tolist()
data_coverage_b2m = pd.DataFrame(data_coverage_list_b2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_a2m = pd.DataFrame(data_coverage_list_a2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_all = pd.DataFrame(data_coverage_list_all).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_before = pd.DataFrame(data_coverage_list_before).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_after = pd.DataFrame(data_coverage_list_after).set_index('loc_id').loc[restaurants_by_4m_coverage]

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
data_coverage_all.loc[restaurants_by_4m_coverage]

In [ ]:
%store restaurants_by_4m_coverage

Visuals

In [ ]:
%store -r time_differences
%store -r time_differences_details

In [ ]:
df.groupby('created_at').agg({'item_quantity' : 'sum'})

In [ ]:
if 'time_differences_details' not in locals() or 'time_differences' not in locals():

    time_differences_details = {}
    time_differences = {}
    for loc_id in restaurants_by_4m_coverage:

        df = sales_and_menu_data[loc_id]

        # Group by transactions (at the same time)
        transactions = df.groupby('created_at').agg({'item_quantity' : 'sum'})

        # Group by individuals days and days of the week
        transaction_by_dayofweek = transactions.groupby([transactions.index.dayofweek, transactions.index.date])

        # Take the index at every group and find the difference between time points (dropping the NaT edges) and convert to hours
        time_diffs_on_dayofweek = transaction_by_dayofweek.apply(lambda s: s.index.to_series().diff().dropna().dt.seconds//3600)

        existing_combinations = time_diffs_on_dayofweek.index.drop_duplicates()

        # Create a new MultiIndex from all days and existing (date, datetime) combinations
        all_days = np.arange(7) 
        new_indices = [(day, date, datetime) for day in all_days for _, date, datetime in existing_combinations]
        time_diffs_on_dayofweek = time_diffs_on_dayofweek.reindex(new_indices)

        # Store entire pivoted data
        time_differences_details[loc_id] = time_diffs_on_dayofweek

        time_diff_frequencies_list = []
        for dayofweek in range(7):

            # Subset to given day of the week and calculate the frequencies
            if dayofweek in time_diffs_on_dayofweek.index.get_level_values(0):
                time_diff_frequencies_specific_day = time_diffs_on_dayofweek[dayofweek].value_counts()
                time_diff_frequencies_specific_day.index.name = "time_diffs"
                time_diff_frequencies_specific_day.name = dayofweek
                time_diff_frequencies_list.append(pd.DataFrame(time_diff_frequencies_specific_day))

        time_diff_frequencies = time_diff_frequencies_list[0].join(time_diff_frequencies_list[1:], how='outer').sort_index()
        time_diff_frequencies = time_diff_frequencies.rename(columns={0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'})
        time_diff_frequencies.columns.name = loc_id

        time_differences[loc_id] = time_diff_frequencies

    %store time_differences
    %store time_differences_details

In [ ]:
true_promos = pd.DataFrame(zip(restaurants_by_4m_coverage, ['Impossible',
                   'Beyond Sausage',
                   'Beyond Burger',
                   'Impossible Sausage',
                   ['Vegan','Bacon'], # and
                   'Vegan Breakfast Sandwich',
                   'Vegan',
                   ['Vegan','Queso'], # and
                   'Beyond',
                   'Kimchee Veggie Burger',
                   'Beyond Sausage',
                   'Wake And Fake',
                   ['Vegan Sausage','Impossible Sausage'], # or
                   'Verde Jackfruit', # (as name not modification)
                   'Beyond Meat Patty Melt',
                   'Vegan Elvis',
                   'Vegan Pattie',
                   'Better Than Beyond',
                   'Impossible',
                   'Impossible',
                   'Beyond Burger',
                   'Veggie Sausage',
                   'Beyond',
                   'Vegan',
                   'Beyond', # N/A
                   'Arkie Vegan',
                   'Impossible Orbit',
                   'Vegan Burger',
                   'Impossible Burger',
                   'Impossible Burger']), columns = ['location_id','promo_name']).set_index('location_id')

before_after_details_true = (before_after_details
                             .join(true_promos, how='left')
                             .assign(cross_over_date = lambda df: df['cross_over_date']
                                     .mask(df.index == 'S8MT0YGD2KTN9', pd.Timestamp('2019-03-11 00:00:00-04:00'))
                                     .mask(df.index == 'V3Q26BHF3SE2H', pd.Timestamp('2021-03-24 00:00:00-05:00'))
                                     .mask(df.index == 'ED5J990H5VAZT', pd.Timestamp('2021-10-01 00:00:00-05:00')))
                             .loc[restaurants_by_4m_coverage]
                             )
%store before_after_details_true

In [ ]:
pd.options.display.float_format = '{:,.10f}'.format

# Turn off auto display
plt.ioff()

freq1 = 'D'
freq2 = 'W'
max_ylim1 = 0
max_ylim2 = 0
max_ylim3 = 0

for i, loc_id in enumerate(restaurants_by_4m_coverage):

    # Relevant DF
    df = sales_and_menu_data[loc_id]
    promo_list = before_after_details_true.loc[loc_id, 'promo_name']

    # Subset
    promo_datetime = pd.to_datetime(before_after_details_true.loc[loc_id, 'cross_over_date']).tz_convert(timezones[loc_id])
    two_months_before = promo_datetime - pd.DateOffset(months=2)
    two_months_after = promo_datetime + pd.DateOffset(months=2)
    period = df.loc[two_months_before:two_months_after]

    # Calculate max y limit for plotting
    all_items1 = period.resample(freq1)['item_quantity'].sum()
    max_ylim1 = max(max_ylim1, all_items1.max())

    # Calculate max y limit for plotting
    all_items2 = period.resample(freq2)['item_quantity'].sum()
    max_ylim2 = max(max_ylim2, all_items2.max())

    # Calculate max y limit for plotting
    all_items3 = df.resample(freq2)['item_quantity'].sum()
    max_ylim3 = max(max_ylim3, all_items3.max())

    gaps_binarized = pd.DataFrame([time_differences[loc_id].iloc[0,].fillna(0), time_differences[loc_id].iloc[1:,].fillna(0).sum(axis=0)], index=['Gaps Less than an Hour', 'Gaps More than an Hour']).astype(int)

    # Summary stats
    print('\n\n\n\n')
    display(Markdown(f'## {loc_id}'))
    
    
    print(f'\nPlant-Based Analog: {promo_list}')
    if loc_id == 'ED5J990H5VAZT':
        promo_item_containing = df[(df['item_name'].str.contains(before_after_details_true.locpromo_list[0]) & 
                                                            df['item_name'].str.contains(promo_list[1]) | 
                                                            df['item_modifications'].str.contains(promo_list[0]) &
                                                            df['item_modifications'].str.contains(promo_list[1]))]
    elif loc_id == 'AQD04SM0J92WA':
        promo_item_containing = df[(df['item_name'].str.contains(promo_list[0]) & 
                                                            df['item_name'].str.contains(promo_list[1]) | 
                                                            df['item_modifications'].str.contains(promo_list[0]) &
                                                            df['item_modifications'].str.contains(promo_list[1]))]
    elif loc_id == 'LBMCPAYT7W36V':
        promo_item_containing = df[(df['item_name'].str.contains(promo_list[0]) | 
                                                            df['item_name'].str.contains(promo_list[1]) | 
                                                            df['item_modifications'].str.contains(promo_list[0]) |
                                                            df['item_modifications'].str.contains(promo_list[1]))]
    elif loc_id == 'SAFK7ND1HR6XS':
        promo_item_containing = df[df['item_name'].str.contains(promo_list)]
    else:
        promo_item_containing = df[df['item_name'].str.contains(promo_list) | df['item_modifications'].str.title().str.contains(promo_list)]
    print(promo_item_containing['item_name'].unique().tolist())
    
    plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
    plt.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')
    plt.title("Plant-Based Analog")
    plt.xticks(rotation=70)

    print(f"\n\n{locations.query('location_id == @loc_id')[['cuisine', 'city', 'state', 'restaurant_type']].to_markdown()}")
    print(f'\n{data_coverage_before.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_all.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_after.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_b2m.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_4m.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_a2m.loc[[loc_id]].to_markdown()}')
    print(f'\n{gaps_binarized.to_markdown()}')
    print(f'\n{df["item_name"].value_counts().sort_values(ascending=False).iloc[:40].to_frame().to_markdown()}')

    # Daily
    total_items = time_differences[loc_id].sum().sum()
    colorbar_max = total_items / 10**3
    plot_time_spacing(loc_id, time_differences, colorbar_max)
    plot_time_series(loc_id, df, before_after_details_true, max_ylim=max_ylim1, freq=freq1)
    plot_time_series(loc_id, df, before_after_details_true, max_ylim=max_ylim2, freq=freq2)
    plot_time_series(loc_id, df, before_after_details_true, max_ylim=max_ylim3, freq=freq2, subset=False)

    # Show
    plt.show()

In [ ]:
plot_time_series('S8MT0YGD2KTN9', sales_and_menu_data['S8MT0YGD2KTN9'], before_after_details_true.assign(cross_over_date = before_after_details_true), max_ylim=max_ylim1, freq=freq1)